#run this command as needed to re-establish connection to Big Query
gcloud auth login --update-adc

In [3]:
import vertexai
import pandas as pd
import numpy as np
import time
from google.cloud import bigquery
import matplotlib.pyplot as plt
import random
random.seed(35)
np.random.seed(35)
client = bigquery.Client()
vertexai.init(project="anbc-hcb-dev", location="us-east4")

In [10]:
sql = """
SELECT * FROM `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_pre_descriptives`
"""
df = client.query(sql).to_dataframe() 
df.head()
# df.shape

(2542308, 18)

Member month = 

In [12]:
df2 = df[df['tenure_yr1'] >= 1].copy()
df2['sum_paid_amt_pmpm'] = df2['sum_paid_amt'] / df2['tenure_yr1']
df2.head()

,asdb_member_key,index_dt,coa_population_category,ss_cohort,agenbr,tenure_yr1,major_chronic_cnt,sum_paid_amt,inpatient_cost,emergency_cost,outpatient_cost,ed_flag,sum_ed_visits,sum_avoidable,sum_preventable,sum_unnecessary,acute_ip_flag,sum_acute_ip_admits,sum_paid_amt_pmpm
3,352035766,2022-12-01,TANF,Medicaid Kid new 48,0.0,8,0,3740.220000000,2771.390000000,0E-9,968.830000000,0,0,0,0,0,0,0.0,467.527500000
5,72849735,2023-08-01,TANF,Medicaid Kid new 03,0.0,3,1,13651.430000000,8111.740000000,0E-9,5539.690000000,0,0,0,0,0,0,0.0,4550.476666666666666666666667
7,590226414,2023-06-01,TANF,Medicaid Kid new 03,0.0,2,0,131.340000000,0E-9,0E-9,131.340000000,0,0,0,0,0,0,0.0,65.670000000
10,72852186,2023-08-01,Foster,Foster,0.0,2,0,418.980000000,0E-9,0E-9,418.980000000,0,0,0,0,0,0,0.0,209.490000000
11,520502241,2023-08-01,TANF,Medicaid Kid new 03,0.0,1,0,81.590000000,0E-9,0E-9,81.590000000,0,0,0,0,0,0,0.0,81.590000000


In [ ]:
#two ways to do PTPY: Individual or group
#individual you'd create a variable (ex sum_paid_amt_pmpm) using the formula: sum_paid_amt/tenure_yr1
#group where you use the formula sum(sum_paid_amt_pmpm)/sum(tenure_yr1)

In [6]:
# (worse IMO): less precise, but you can calculate at the individual level first
df2.groupby(['ss_cohort'])['sum_paid_amt_pmpm'].agg('sum')/df2.groupby('ss_cohort').size()

ss_cohort
ABD                      730.0073480668431303289981369
ABD new                  2999.622446205098056949908801
BH                       1280.559683588062941296772122
Dual                     534.4712767761326842196702940
Foster                   660.1600581564668451662055004
LTSS                     4428.959429451232981415467582
Medicaid Adult           284.9181215321347635550953522
Medicaid Adult new 03    2312.928626641279889221334783
Medicaid Adult new 48    752.9843861305584806044029366
Medicaid KID             129.8392463315409404267668465
Medicaid Kid new 03      1528.065972424923359665176347
Medicaid Kid new 48      685.3947064858607359204038310
dtype: object

In [7]:
#(better IMO) sum over both pmpm and member months within a given group to get a final group-level PMPM
df2.groupby(['ss_cohort'])['sum_paid_amt'].agg('sum')/df2.groupby(['ss_cohort'])['tenure_yr1'].agg('sum')

ss_cohort
ABD                      726.8169176603641488010555558
ABD new                  2199.199368106069046747417512
BH                       1111.619366918252316307170882
Dual                     489.9370291593127272178727127
Foster                   572.3591388336493548536280812
LTSS                     4145.141224247607026132285977
Medicaid Adult           282.8295196824193898833109584
Medicaid Adult new 03    1813.214319035291804567945735
Medicaid Adult new 48    736.8841606380109260578831467
Medicaid KID             127.9768207365306246470149204
Medicaid Kid new 03      1421.137122988277369362209418
Medicaid Kid new 48      646.5613155540341841775647858
dtype: object

In [8]:
df2['sum_ip_ptpy'] = (df2['sum_acute_ip_admits'] / (df2['tenure_yr1'] / 12)) * 1000

/var/tmp/ipykernel_3266939/1728186023.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['sum_ip_ptpy'] = (df2['sum_acute_ip_admits'] / (df2['tenure_yr1'] / 12)) * 1000


In [17]:
df['ss_cohort'].unique()

array(['Medicaid Kid new 03', 'Medicaid Kid new 48', 'Foster',
       'Medicaid KID', 'Medicaid Adult', 'ABD new', 'ABD', 'LTSS', 'BH',
       'Dual', 'Medicaid Adult new 03', 'Medicaid Adult new 48'],
      dtype=object)

In [40]:
grouped = df2.groupby('ss_cohort')

# Performs aggregate functions on these columns
result = grouped.agg({
    'sum_paid_amt': 'sum',
    'tenure_yr1': 'sum',
    'sum_acute_ip_admits': 'sum',
    'sum_ed_visits': 'sum',
    'ed_flag': 'sum',
    'acute_ip_flag': 'sum',  
    'asdb_member_key': 'nunique',  # Count unique member IDs in each cohort
    'major_chronic_cnt': 'mean' # Automatically takes the mean here
})

# Sum of all sum_paid_amt / sum of all tenure lengths
result['med_paid_amt_pmpm'] = result['sum_paid_amt'] / result['tenure_yr1']

# Sum of all su
result['ed_visits_ptpy'] = (result['sum_ed_visits'] / (result['tenure_yr1'] / 12)) * 1000
result['ip_admits_ptpy'] = (result['sum_acute_ip_admits'] / (result['tenure_yr1'] / 12)) * 1000

# ( sum of flags / num members ) * 100
result['perc_had_ed_visits'] = (result['ed_flag'] / grouped.size()) * 100
result['perc_had_ip_admits'] = (result['acute_ip_flag'] / grouped.size()) * 100


pd.set_option('display.precision', 2)
pd.set_option('display.max_columns', None)  # Ensure all columns are shown
pd.set_option('display.max_rows', None)  # Ensure all columns are shown
specific_cohorts = result.loc[['ABD', 'Medicaid Adult', 'Medicaid KID']]
specific_cohorts[['asdb_member_key','med_paid_amt_pmpm', 'major_chronic_cnt', 'perc_had_ed_visits', 'perc_had_ip_admits','ed_visits_ptpy', 'ip_admits_ptpy',]].transpose()

# Sanity check for adding (outcome of interest, for post IP) stratify on
# Make sure to specify metrics are from pre-period

ss_cohort,ABD,Medicaid Adult,Medicaid KID
asdb_member_key,130659,801010,801096
med_paid_amt_pmpm,726.8169176603641488010555558,282.8295196824193898833109584,127.9768207365306246470149204
major_chronic_cnt,3.37,2.25,0.6
perc_had_ed_visits,33.86,32.04,27.04
perc_had_ip_admits,6.01,2.71,0.9
ed_visits_ptpy,831.62,673.31,449.43
ip_admits_ptpy,94.88,35.85,10.39
